<a href="https://colab.research.google.com/github/almahova/iemocap-proj/blob/data-modelling/iemocap_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IEMOCAP Emotion Recognition — Modelling Pipeline
## Wav2Vec2 + Leave-One-Session-Out Cross-Validation

This notebook implements an **Audio-Only Transformer pipeline** for 3-class Speech Emotion Recognition (SER) on the IEMOCAP corpus.

| Component | Choice |
|-----------|--------|
| Model | `Wav2Vec2ForSequenceClassification` (`facebook/wav2vec2-base`) |
| Validation | Leave-One-Session-Out CV — 5 folds, one held-out session per fold |
| Loss | `CrossEntropyLoss` with per-fold dynamic class weights |
| Primary metric | **Macro F1-Score** (robust to the ~55 % Negative class imbalance) |
| Classes | `Negative` (0) · `Positive` (1) · `Neutral` (2) |

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

CUDA Available: False


In [ ]:
import io
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report
import soundfile as sf
import torchaudio.transforms as T
from tqdm import tqdm
from transformers import get_linear_schedule_with_warmup
import matplotlib.pyplot as plt
import itertools
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

Using device: cpu
PyTorch version: 2.12.0+cu130


## Step 1: Session Extraction & Label Mapping

Extract the integer session number from the `file` column (`Ses01` → `1`, …, `Ses05` → `5`) to use as group IDs for LOSO-CV, and map the 3-class `emotion_group` labels to integers.

In [ ]:
from datasets import load_dataset

# Load IEMOCAP from HuggingFace (mirrors EDA notebook)
ds = load_dataset('AbstractTTS/IEMOCAP')
df = ds['train'].to_pandas()

# Emotion grouping — same mapping as the EDA notebook
emotion_group_map = {
    'happy':      'Positive',
    'excited':    'Positive',
    'angry':      'Negative',
    'frustrated': 'Negative',
    'fear':       'Negative',
    'sad':        'Negative',
    'disgust':    'Negative',
    'other':      'Neutral',
    'neutral':    'Neutral',
    'surprise':   'Neutral',
}
df['emotion_group'] = df['major_emotion'].map(emotion_group_map)

# Extract integer session number (1–5) for LOSO grouping
df['session_num'] = df['file'].str.extract(r'Ses(\d{2})').astype(int)

# Encode emotion_group -> integer label
LABEL_MAP = {'Negative': 0, 'Positive': 1, 'Neutral': 2}
ID2LABEL  = {v: k for k, v in LABEL_MAP.items()}
df['label'] = df['emotion_group'].map(LABEL_MAP)

# Modelling DataFrame: preserve the audio column for the Dataset class
df_model = (
    df[['file', 'audio', 'emotion_group', 'label', 'session_num']]
    .dropna()
    .reset_index(drop=True)
)

print(f'Total records : {len(df_model)}')
print(f'\nLabel distribution:')
print(df_model['emotion_group'].value_counts())
print(f'\nSession distribution:')
print(df_model['session_num'].value_counts().sort_index())

Total records : 10039

Label distribution:
emotion_group
Negative    5545
Positive    2632
Neutral     1862
Name: count, dtype: int64

Session distribution:
session_num
1    1819
2    1811
3    2136
4    2103
5    2170
Name: count, dtype: int64


## Step 2: Custom PyTorch Dataset & Audio Preprocessing

`IEMOCAPDataset` handles four responsibilities:

1. **Loading** raw audio from the HuggingFace `bytes` dict via `soundfile`.
2. **Mono conversion** — any stereo recordings are averaged to one channel.
3. **Resampling** to 16 000 Hz using `torchaudio.transforms.Resample` (Wav2Vec2 requires exactly 16 kHz).
4. **Feature extraction** via `Wav2Vec2Processor`: zero-mean unit-variance normalisation plus padding / truncation to a fixed `max_duration`.

> No hand-crafted MFCC or spectrogram features are needed — the processor works directly on the raw float waveform.

In [ ]:
CHECKPOINT   = 'facebook/wav2vec2-base'
MAX_DURATION = 8.0     # seconds — covers ~98 % of IEMOCAP utterances
TARGET_SR    = 16_000  # Hz — required by all Wav2Vec2 variants


class IEMOCAPDataset(Dataset):
    # PyTorch Dataset for IEMOCAP audio emotion classification.
    # Each __getitem__ returns a dict with input_values, attention_mask, and labels.

    def __init__(self, df, processor, max_duration=MAX_DURATION, target_sr=TARGET_SR):
        self.df         = df.reset_index(drop=True)
        self.processor  = processor
        self.max_length = int(max_duration * target_sr)  # 128 000 samples at 8 s
        self.target_sr  = target_sr

    def __len__(self):
        return len(self.df)

    def _load_audio(self, audio_data):
        # Return (waveform float32 ndarray, sampling_rate) from a HuggingFace audio dict.
        if isinstance(audio_data, dict):
            raw_bytes = audio_data.get('bytes')
            if raw_bytes is None:
                raise ValueError('Audio dict is missing the bytes key.')
            waveform, sr = sf.read(io.BytesIO(raw_bytes), dtype='float32')
        elif isinstance(audio_data, np.ndarray):
            waveform = audio_data.astype(np.float32)
            sr = self.target_sr
        else:
            raise TypeError(f'Unsupported audio type: {type(audio_data)}')

        # Stereo -> mono by averaging channels
        if waveform.ndim > 1:
            waveform = waveform.mean(axis=-1)

        return waveform, sr

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        waveform, sr = self._load_audio(row['audio'])

        # Resample to TARGET_SR if the stored rate differs
        if sr != self.target_sr:
            waveform_t = torch.from_numpy(waveform).unsqueeze(0)
            resampler  = T.Resample(orig_freq=sr, new_freq=self.target_sr)
            waveform   = resampler(waveform_t).squeeze(0).numpy()

        # Normalise + pad/truncate to a fixed sequence length
        encoded = self.processor(
            waveform,
            sampling_rate=self.target_sr,
            return_tensors='pt',
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
        )

        return {
            'input_values' : encoded['input_values'].squeeze(0),           # (max_length,)
            'attention_mask': encoded.get(
                'attention_mask', torch.ones(self.max_length)
            ).squeeze(0),                                                   # (max_length,)
            'labels': torch.tensor(row['label'], dtype=torch.long),        # scalar
        }

In [ ]:
# Quick sanity check — verify tensor shapes before training
_processor = Wav2Vec2Processor.from_pretrained(CHECKPOINT)
_check_ds  = IEMOCAPDataset(df_model.head(2), _processor)
_sample    = _check_ds[0]

print('input_values  shape:', _sample['input_values'].shape)   # expect (128000,)
print('attention_mask shape:', _sample['attention_mask'].shape) # expect (128000,)
print('label              :', _sample['labels'].item(),
      '->', ID2LABEL[_sample['labels'].item()])

del _check_ds, _sample

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

input_values  shape: torch.Size([128000])
attention_mask shape: torch.Size([128000])
label              : 2 -> Neutral


## Step 3: Leave-One-Session-Out (LOSO) Cross-Validation Splitter

`LeaveOneGroupOut` from scikit-learn creates 5 folds where the **test set contains exactly one complete IEMOCAP session**. Because each session features a unique male/female speaker pair who never appears in any other session, this guarantees complete **speaker independence** between train and test — the gold standard for SER evaluation and the correct way to avoid *Speaker Leakage*.

| Fold | Test session | Train sessions |
|------|-------------|----------------|
| 1 | Ses01 | 2, 3, 4, 5 |
| 2 | Ses02 | 1, 3, 4, 5 |
| 3 | Ses03 | 1, 2, 4, 5 |
| 4 | Ses04 | 1, 2, 3, 5 |
| 5 | Ses05 | 1, 2, 3, 4 |

In [ ]:
logo   = LeaveOneGroupOut()
groups = df_model['session_num'].values

print(f'LOSO-CV: {logo.get_n_splits(groups=groups)} folds\n')

for fold_idx, (train_idx, val_idx) in enumerate(logo.split(df_model, groups=groups)):
    held        = df_model.iloc[val_idx]['session_num'].iloc[0]
    train_sess  = sorted(df_model.iloc[train_idx]['session_num'].unique())
    label_dist  = df_model.iloc[val_idx]['emotion_group'].value_counts().to_dict()
    print(
        f'Fold {fold_idx + 1}  |  Test: Ses0{held}  |  '
        f'Train sessions: {train_sess}  |  '
        f'Val size: {len(val_idx):>4d}  |  {label_dist}'
    )

LOSO-CV: 5 folds

Fold 1  |  Test: Ses01  |  Train sessions: [np.int64(2), np.int64(3), np.int64(4), np.int64(5)]  |  Val size: 1819  |  {'Negative': 947, 'Positive': 446, 'Neutral': 426}
Fold 2  |  Test: Ses02  |  Train sessions: [np.int64(1), np.int64(3), np.int64(4), np.int64(5)]  |  Val size: 1811  |  {'Negative': 909, 'Positive': 518, 'Neutral': 384}
Fold 3  |  Test: Ses03  |  Train sessions: [np.int64(1), np.int64(2), np.int64(4), np.int64(5)]  |  Val size: 2136  |  {'Negative': 1273, 'Positive': 506, 'Neutral': 357}
Fold 4  |  Test: Ses04  |  Train sessions: [np.int64(1), np.int64(2), np.int64(3), np.int64(5)]  |  Val size: 2103  |  {'Negative': 1267, 'Positive': 549, 'Neutral': 287}
Fold 5  |  Test: Ses05  |  Train sessions: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]  |  Val size: 2170  |  {'Negative': 1149, 'Positive': 613, 'Neutral': 408}


## Steps 4 & 5: Class Weights · Model Initialisation · Training Loop

**Step 4 — Per-fold class weights (dynamic):**
Class weights are computed *only from the current fold's training set* using `sklearn.utils.class_weight.compute_class_weight('balanced', ...)`, which implements:

$$weight_c = \frac{N}{C \cdot n_c}$$

where $N$ = total training samples, $C$ = number of classes, $n_c$ = samples in class $c$.

These weights are passed directly to `nn.CrossEntropyLoss(weight=...)`, so the model incurs a proportionally higher penalty for misclassifying the minority Positive / Neutral classes — preventing the Negative blind-guess bias. Computing weights **per fold** ensures they reflect only the available training data, not the held-out session.

**Step 5 — Model + Training:**
`Wav2Vec2ForSequenceClassification.from_pretrained(...)` loads the pre-trained wav2vec2 encoder from HuggingFace and attaches a fresh 3-way classification head. The entire model is fine-tuned end-to-end. Gradient clipping (`max_norm=1.0`) stabilises training on the variable-length emotion recordings.

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
CHECKPOINT    = 'facebook/wav2vec2-base'
NUM_LABELS    = 3
BATCH_SIZE    = 8      # reduce to 4 if GPU runs out of memory
NUM_EPOCHS    = 8
LEARNING_RATE = 2e-5
MAX_DURATION  = 4.0    # seconds — halves compute vs 8.0; covers most utterances
PATIENCE      = 3      # early stopping — stop if no improvement for 3 epochs
NUM_WORKERS   = 2

In [ ]:
def train_epoch(model, loader, optimiser, loss_fn, device, scheduler=None):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []

    for batch in tqdm(loader, desc='  train', leave=False):
        input_values   = batch['input_values'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimiser.zero_grad()
        logits = model(input_values=input_values, attention_mask=attention_mask).logits
        loss   = loss_fn(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        all_preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, macro_f1


def eval_epoch(model, loader, loss_fn, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc='  val  ', leave=False):
            input_values   = batch['input_values'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            logits = model(input_values=input_values, attention_mask=attention_mask).logits
            loss   = loss_fn(logits, labels)

            total_loss += loss.item()
            all_preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, macro_f1


def eval_epoch_with_preds(model, loader, loss_fn, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc='  eval ', leave=False):
            input_values   = batch['input_values'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            logits = model(input_values=input_values, attention_mask=attention_mask).logits
            loss   = loss_fn(logits, labels)

            total_loss += loss.item()
            all_preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, macro_f1, all_preds, all_labels

In [ ]:
fold_results = []
sep = '=' * 60

for fold_idx, (train_idx, val_idx) in enumerate(logo.split(df_model, groups=groups)):
    held_session = df_model.iloc[val_idx]['session_num'].iloc[0]
    print(f'\n{sep}')
    print(f'FOLD {fold_idx + 1}/5  |  Held-out session: Ses0{held_session}')
    print(f'  Train: {len(train_idx)} samples  |  Val: {len(val_idx)} samples')

    train_df = df_model.iloc[train_idx].reset_index(drop=True)
    val_df   = df_model.iloc[val_idx].reset_index(drop=True)

    # ── Step 4: compute class weights from this fold's training set only ──
    train_labels  = train_df['label'].values
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.array([0, 1, 2]),
        y=train_labels
    )
    weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
    loss_fn       = nn.CrossEntropyLoss(weight=weight_tensor)

    print(f'  Class weights -> '
          f'Negative: {class_weights[0]:.3f} | '
          f'Positive: {class_weights[1]:.3f} | '
          f'Neutral : {class_weights[2]:.3f}')

    # ── Datasets & DataLoaders ────────────────────────────────────────────
    processor     = Wav2Vec2Processor.from_pretrained(CHECKPOINT)
    train_dataset = IEMOCAPDataset(train_df, processor, max_duration=MAX_DURATION)
    val_dataset   = IEMOCAPDataset(val_df,   processor, max_duration=MAX_DURATION)

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE,
        shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE,
        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
    )

    # ── Step 5: fresh model + freeze feature encoder ──────────────────────
    model = Wav2Vec2ForSequenceClassification.from_pretrained(
        CHECKPOINT,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL_MAP,
        ignore_mismatched_sizes=True,
    ).to(device)

    model.freeze_feature_encoder()
    print(f'  Feature encoder frozen.')

    optimiser    = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps  = len(train_loader) * NUM_EPOCHS
    warmup_steps = max(1, int(0.1 * total_steps))
    scheduler    = get_linear_schedule_with_warmup(optimiser, warmup_steps, total_steps)
    best_val_f1       = 0.0
    best_preds        = []
    best_labels_fold  = []
    epochs_no_improve = 0

    # ── Training loop for this fold ───────────────────────────────────────
    for epoch in range(NUM_EPOCHS):
        tr_loss, tr_f1 = train_epoch(model, train_loader, optimiser, loss_fn, device, scheduler)
        vl_loss, vl_f1 = eval_epoch(model, val_loader, loss_fn, device)

        print(f'  Ep {epoch + 1}/{NUM_EPOCHS}  |  '
              f'Train loss: {tr_loss:.4f}  F1: {tr_f1:.4f}  |  '
              f'Val loss: {vl_loss:.4f}  F1: {vl_f1:.4f}')

        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            torch.save(model.state_dict(), f'best_model_fold{fold_idx + 1}.pt')
            _, _, best_preds, best_labels_fold = eval_epoch_with_preds(
                model, val_loader, loss_fn, device
            )
            print(f'    >> New best saved  (Val Macro F1 = {best_val_f1:.4f})')
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f'  Early stopping triggered at epoch {epoch + 1}')
                break

    fold_results.append({
        'fold'        : fold_idx + 1,
        'test_session': f'Ses0{held_session}',
        'best_val_f1' : round(best_val_f1, 4),
        'preds'       : best_preds,
        'labels'      : best_labels_fold,
    })

    # Free GPU memory before the next fold
    del model, train_dataset, val_dataset, train_loader, val_loader
    torch.cuda.empty_cache()


FOLD 1/5  |  Held-out session: Ses01
  Train: 8220 samples  |  Val: 1819 samples
  Class weights -> Negative: 0.596 | Positive: 1.253 | Neutral : 1.908


pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
projector.bias               | MISSING    | 
projector.weight             | MISSING    | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
  train:   0%|          | 0/1028 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
# ── LOSO-CV Results Summary ───────────────────────────────────────────────────
results_df = pd.DataFrame(fold_results)

print('\n' + '=' * 60)
print('LOSO-CV FINAL RESULTS')
print('=' * 60)
print(results_df.to_string(index=False))
print(f'\nMean Macro F1 : {results_df["best_val_f1"].mean():.4f}')
print(f'Std  Macro F1 : {results_df["best_val_f1"].std():.4f}')
print(f'\nBest checkpoints: best_model_fold1.pt ... best_model_fold5.pt')

## Results Visualisation

Confusion matrix (aggregated across all 5 LOSO folds) and per-class F1 breakdown.

In [ ]:
import itertools
from sklearn.metrics import confusion_matrix, classification_report

# Aggregate predictions across all folds
all_preds_agg  = list(itertools.chain.from_iterable(r['preds']  for r in fold_results))
all_labels_agg = list(itertools.chain.from_iterable(r['labels'] for r in fold_results))

class_names = ['Negative', 'Positive', 'Neutral']

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm      = confusion_matrix(all_labels_agg, all_preds_agg)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
axes[0].set_title('Confusion Matrix — Raw Counts', fontsize=13, weight='bold')
plt.colorbar(im0, ax=axes[0])
for i, j in itertools.product(range(3), range(3)):
    axes[0].text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=12)
axes[0].set_xticks(range(3)); axes[0].set_xticklabels(class_names)
axes[0].set_yticks(range(3)); axes[0].set_yticklabels(class_names)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

im1 = axes[1].imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix — Normalised', fontsize=13, weight='bold')
plt.colorbar(im1, ax=axes[1])
for i, j in itertools.product(range(3), range(3)):
    axes[1].text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center', fontsize=12)
axes[1].set_xticks(range(3)); axes[1].set_xticklabels(class_names)
axes[1].set_yticks(range(3)); axes[1].set_yticklabels(class_names)
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.tight_layout()
plt.show()

# ── Per-class F1 per fold ─────────────────────────────────────────────────────
fold_class_f1 = []
for r in fold_results:
    report = classification_report(r['labels'], r['preds'],
                                   target_names=class_names, output_dict=True, zero_division=0)
    fold_class_f1.append({cls: report[cls]['f1-score'] for cls in class_names})

f1_df = pd.DataFrame(fold_class_f1, index=[f'Fold {r["fold"]}' for r in fold_results])

fig, ax = plt.subplots(figsize=(10, 5))
f1_df.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white')
ax.set_title('Per-Class F1 Score by Fold', fontsize=13, weight='bold')
ax.set_xlabel('Fold'); ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1)
ax.axhline(y=0.33, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
ax.legend(); ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

# ── Full classification report ────────────────────────────────────────────────
print('\nAggregated Classification Report (all folds):')
print(classification_report(all_labels_agg, all_preds_agg,
                             target_names=class_names, zero_division=0))